# verify_dbms — Phi-4 Multimodal (HuggingFace Transformers)

Runs **microsoft/Phi-4-multimodal-instruct** locally inside Colab to answer questions from `dbms_dataset_mixed.json`.

- All 8 question types supported (Multiple Choice, Short Answer, SQL Interpretation, etc.)
- Images loaded from an uploaded `images/` zip
- Answers written to `/content/outputs/dbms/<id>.txt`
- Already-answered entries are skipped on re-run

> **Runtime:** A100 GPU (Colab Pro) strongly recommended.

In [ ]:
!pip install -q transformers accelerate pillow torch torchvision

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Paste your HuggingFace token: ").strip()

login(token=hf_token)
print("Logged in to HuggingFace.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"

print("Loading processor…")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (may take a few minutes)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

In [ ]:
import os, zipfile
from google.colab import files

# Upload JSON
print("Upload dbms_dataset_mixed.json")
uploaded = files.upload()
JSON_PATH = list(uploaded.keys())[0]
print("JSON:", JSON_PATH)

# Upload images zip
IMAGES_DIR = "/content/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

print("\nUpload images.zip (zip your 'images/' folder)")
img_uploaded = files.upload()
zip_name = list(img_uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(IMAGES_DIR)

print(f"Extracted images to {IMAGES_DIR}")
print("Sample:", os.listdir(IMAGES_DIR)[:5])

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/content/outputs/dbms"
MAX_NEW_TOKENS  = 1024

SYSTEM_PROMPT = (
    "You are an expert database systems tutor. "
    "Analyze any provided diagram or image carefully before answering. "
    "Answer step by step, showing your reasoning clearly. "
    "For multiple-choice questions, state the correct option letter and explain why."
)

FILTER_IDS  = None   # e.g. ["dbms_001", "dbms_002"]
FILTER_TYPE = None   # e.g. "Multiple Choice"
LAST_N      = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json
from PIL import Image

def load_entries(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def build_prompt_text(entry):
    q_type = entry.get("question_type", "")
    lines  = []
    if q_type:
        lines.append(f"[Question Type: {q_type}]")
        lines.append("")
    lines.append(entry["question"].strip())
    choices = entry.get("choices") or {}
    if choices:
        lines.append("")
        lines.append("Choices:")
        for letter, text in sorted(choices.items()):
            lines.append(f"  {letter}. {text}")
    return "\n".join(lines)

def resolve_image(entry):
    rel = entry.get("image", "")
    if not rel:
        return None
    filename = os.path.basename(rel)
    full = os.path.join(IMAGES_DIR, filename)
    return full if os.path.exists(full) else None

def resize_to_224(img):
    w, h   = img.size
    side   = max(w, h)
    padded = Image.new("RGB", (side, side), (255, 255, 255))
    padded.paste(img, ((side - w) // 2, (side - h) // 2))
    return padded.resize((224, 224), Image.LANCZOS)

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    question_text = build_prompt_text(entry)
    pil_images    = []

    img_path = resolve_image(entry)
    if img_path:
        pil_images.append(resize_to_224(Image.open(img_path).convert("RGB")))
    elif entry.get("image"):
        print(f"  [WARN] #{entry['id']} — image not found: {entry['image']}")

    image_tags  = "".join(f"<|image_{i+1}|>" for i in range(len(pil_images)))
    phi4_prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{image_tags}{question_text}<|end|>\n"
        f"<|assistant|>\n"
    )

    inputs = processor(
        text=phi4_prompt,
        images=pil_images if pil_images else None,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer     = processor.decode(new_tokens, skip_special_tokens=True).strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)

    q_type = entry.get("question_type", "")
    print(f"  [OK]   #{entry['id']} [{q_type}] → {out_path}")

print("Helpers defined.")

In [ ]:
all_entries = load_entries(JSON_PATH)

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

if FILTER_TYPE:
    entries = [e for e in entries if e.get("question_type", "").lower() == FILTER_TYPE.lower()]

print(f"Model  : {MODEL_ID}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
for entry in entries:
    try:
        ask(entry)
    except Exception as exc:
        import traceback
        print(f"  [ERR]  #{entry['id']} — {type(exc).__name__}: {exc}")
        traceback.print_exc()

print("\nDone.")

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/dbms_phi4_answers"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")